# 01 Market Overview

A-share market daily kline overview using DuckDB + Plotly.

This notebook:
1. Scans `data/kline/**/*.parquet` using DuckDB for high-performance multi-file queries
2. Shows bar counts by symbol to understand data coverage
3. Renders an interactive candlestick chart for a selected stock

In [1]:
import sys
import os
from pathlib import Path

# Adjust path so trade_py is importable when running from repo root
repo_root = Path(os.getcwd()).parent if Path(os.getcwd()).name == 'notebooks' else Path(os.getcwd())
python_path = repo_root / 'python'
if str(python_path) not in sys.path:
    sys.path.insert(0, str(python_path))

DATA_ROOT = repo_root / 'data'
KLINE_GLOB = str(DATA_ROOT / 'kline' / '**' / '*.parquet')
print(f'Repo root : {repo_root}')
print(f'Kline glob: {KLINE_GLOB}')

Repo root : /data00/home/guohuanwei.cztj/git_files/trade
Kline glob: /data00/home/guohuanwei.cztj/git_files/trade/data/kline/**/*.parquet


In [2]:
import duckdb
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px

con = duckdb.connect()
print('DuckDB version:', duckdb.__version__)

DuckDB version: 1.4.4


## 1. Data Coverage: Bar Count by Symbol

In [3]:
try:
    coverage_df = con.execute(f"""
        SELECT
            symbol,
            COUNT(*) AS bar_count,
            MIN(date) AS first_date,
            MAX(date) AS last_date
        FROM read_parquet('{KLINE_GLOB}', union_by_name=true)
        GROUP BY symbol
        ORDER BY bar_count DESC
    """).df()
    print(f'Total symbols: {len(coverage_df)}')
    print(f'Total bars   : {coverage_df["bar_count"].sum():,}')
    coverage_df.head(20)
except Exception as e:
    print(f'No kline data found at {KLINE_GLOB}: {e}')
    coverage_df = pd.DataFrame(columns=['symbol', 'bar_count', 'first_date', 'last_date'])

Total symbols: 146
Total bars   : 56,879


In [4]:
if not coverage_df.empty:
    top30 = coverage_df.head(30)
    fig = px.bar(
        top30,
        x='symbol',
        y='bar_count',
        title='Top 30 Symbols by Bar Count',
        labels={'bar_count': 'Trading Days', 'symbol': 'Symbol'},
        color='bar_count',
        color_continuous_scale='Blues',
    )
    fig.update_layout(xaxis_tickangle=-45, height=450)
    fig.show()
else:
    print('No data to plot. Run data ingestion first.')

## 2. Date Range Distribution

In [5]:
if not coverage_df.empty:
    try:
        daily_counts = con.execute(f"""
            SELECT
                CAST(date AS VARCHAR) AS trade_date,
                COUNT(DISTINCT symbol) AS active_symbols,
                COUNT(*) AS total_bars
            FROM read_parquet('{KLINE_GLOB}', union_by_name=true)
            GROUP BY date
            ORDER BY date
        """).df()

        fig = go.Figure()
        fig.add_trace(go.Scatter(
            x=daily_counts['trade_date'],
            y=daily_counts['active_symbols'],
            mode='lines',
            name='Active Symbols',
            line=dict(color='steelblue', width=1.5),
        ))
        fig.update_layout(
            title='Daily Active Symbol Count (Data Coverage)',
            xaxis_title='Date',
            yaxis_title='Symbol Count',
            height=350,
        )
        fig.show()
    except Exception as e:
        print(f'Query error: {e}')

## 3. Candlestick Chart for a Single Stock

Change `SYMBOL` to any A-share code present in your data.

In [6]:
# Select the first symbol from coverage if available, otherwise use default
SYMBOL = coverage_df['symbol'].iloc[0] if not coverage_df.empty else '600000.SH'
LOOKBACK_DAYS = 120  # trading days to display
print(f'Plotting candlestick for: {SYMBOL}')

Plotting candlestick for: 000006.SZ


In [7]:
try:
    bars_df = con.execute(f"""
        SELECT
            CAST(date AS VARCHAR)  AS date,
            open, high, low, close,
            volume, amount,
            turnover_rate,
            prev_close,
            vwap
        FROM read_parquet('{KLINE_GLOB}', union_by_name=true)
        WHERE symbol = '{SYMBOL}'
        ORDER BY date DESC
        LIMIT {LOOKBACK_DAYS}
    """).df()
    bars_df = bars_df.sort_values('date').reset_index(drop=True)
    print(f'Loaded {len(bars_df)} bars for {SYMBOL}')
    bars_df.tail(5)
except Exception as e:
    print(f'Failed to load bars for {SYMBOL}: {e}')
    bars_df = pd.DataFrame()

Loaded 120 bars for 000006.SZ


In [8]:
if not bars_df.empty:
    # Compute change pct for bar color
    bars_df['change_pct'] = (bars_df['close'] - bars_df['prev_close']) / bars_df['prev_close'].replace(0, float('nan')) * 100

    fig = go.Figure()

    # Candlestick
    fig.add_trace(go.Candlestick(
        x=bars_df['date'],
        open=bars_df['open'],
        high=bars_df['high'],
        low=bars_df['low'],
        close=bars_df['close'],
        name=SYMBOL,
        increasing_line_color='#e84545',   # A-share: red = up
        decreasing_line_color='#2ba02b',   # A-share: green = down
    ))

    # VWAP overlay
    if 'vwap' in bars_df.columns and bars_df['vwap'].notna().any():
        fig.add_trace(go.Scatter(
            x=bars_df['date'],
            y=bars_df['vwap'],
            mode='lines',
            name='VWAP',
            line=dict(color='orange', width=1, dash='dot'),
        ))

    fig.update_layout(
        title=f'{SYMBOL} - Daily Candlestick (Last {LOOKBACK_DAYS} Days)',
        xaxis_title='Date',
        yaxis_title='Price (CNY)',
        xaxis_rangeslider_visible=False,
        height=500,
        legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
    )
    fig.show()

    # Volume subplot
    colors = ['#e84545' if c >= 0 else '#2ba02b' for c in bars_df['change_pct'].fillna(0)]
    vol_fig = go.Figure(go.Bar(
        x=bars_df['date'],
        y=bars_df['volume'],
        marker_color=colors,
        name='Volume',
    ))
    vol_fig.update_layout(
        title=f'{SYMBOL} - Volume',
        xaxis_title='Date',
        yaxis_title='Shares',
        height=250,
    )
    vol_fig.show()
else:
    print(f'No data for {SYMBOL}. Check your data directory.')

## 4. Summary Statistics

In [9]:
if not bars_df.empty:
    stats = {
        'Symbol': SYMBOL,
        'Period': f"{bars_df['date'].iloc[0]} to {bars_df['date'].iloc[-1]}",
        'Trading Days': len(bars_df),
        'Latest Close': f"{bars_df['close'].iloc[-1]:.2f}",
        'Period High': f"{bars_df['high'].max():.2f}",
        'Period Low': f"{bars_df['low'].min():.2f}",
        'Avg Daily Volume': f"{bars_df['volume'].mean():,.0f}",
        'Avg Turnover Rate': f"{bars_df['turnover_rate'].mean():.4f}",
    }
    for k, v in stats.items():
        print(f'{k:25s}: {v}')

con.close()

Symbol                   : 000006.SZ
Period                   : 2025-08-29 to 2026-03-04
Trading Days             : 120
Latest Close             : 390.37
Period High              : 651.34
Period Low               : 329.42
Avg Daily Volume         : 507,806
Avg Turnover Rate        : 3.7616
